# 02 — Silver Transformation

Cleans and enriches the Bronze data:
- Type-casts all columns
- Fills missing publishers with 'Unknown'
- Adds marketing-focused derived columns (sales tier, regional %, global hit flag)
- Partitions output by Genre for query performance

In [ ]:
# Read from Bronze
df_b = spark.table('gaming_bronze.vg_sales_raw')
print(f'Bronze row count: {df_b.count()}')
display(df_b.limit(5))

In [ ]:
from pyspark.sql.functions import col, count, when, isnan

# Count nulls per business column
biz_cols = ['Name','Platform','Year_of_Release','Genre','Publisher',
            'NA_Sales','EU_Sales','JP_Sales','Other_Sales','Global_Sales']
null_counts = df_b.select([
    count(when(col(c).isNull(), c)).alias(c) for c in biz_cols
])
display(null_counts)

In [ ]:
from pyspark.sql.functions import trim, lit
from pyspark.sql.types import IntegerType, DoubleType

df_silver = (
    df_b
    .filter(col('Name').isNotNull())
    .filter(col('Global_Sales').isNotNull())
    .filter(col('Global_Sales').cast(DoubleType()) > 0)
    .withColumn('game_name',  trim(col('Name')))
    .withColumn('platform',   trim(col('Platform')))
    .withColumn('genre',      trim(col('Genre')))
    .withColumn('publisher',  when(col('Publisher').isNull(), lit('Unknown'))
                             .otherwise(trim(col('Publisher'))))
    .withColumn('year_of_release', col('Year_of_Release').cast(IntegerType()))
    .withColumn('na_sales_millions',    col('NA_Sales').cast(DoubleType()))
    .withColumn('eu_sales_millions',    col('EU_Sales').cast(DoubleType()))
    .withColumn('jp_sales_millions',    col('JP_Sales').cast(DoubleType()))
    .withColumn('other_sales_millions', col('Other_Sales').cast(DoubleType()))
    .withColumn('global_sales_millions',col('Global_Sales').cast(DoubleType()))
    .drop('Rank','Name','Platform','Genre','Publisher',
          'Year_of_Release','NA_Sales','EU_Sales','JP_Sales',
          'Other_Sales','Global_Sales',
          '_ingested_at','_source_file','_layer')
)
print(f'Silver row count: {df_silver.count()}')
display(df_silver.limit(5))

In [ ]:
from pyspark.sql.functions import round as spark_round

df_silver = (
    df_silver
    .withColumn('na_sales_pct',
        spark_round(col('na_sales_millions') / col('global_sales_millions') * 100, 1))
    .withColumn('eu_sales_pct',
        spark_round(col('eu_sales_millions') / col('global_sales_millions') * 100, 1))
    .withColumn('jp_sales_pct',
        spark_round(col('jp_sales_millions') / col('global_sales_millions') * 100, 1))
    .withColumn('is_global_hit',
        when(col('global_sales_millions') >= 1.0, True).otherwise(False))
    .withColumn('sales_tier',
        when(col('global_sales_millions') >= 10, 'Blockbuster')
        .when(col('global_sales_millions') >= 1,  'Hit')
        .when(col('global_sales_millions') >= 0.1,'Mid-Tier')
        .otherwise('Long-Tail'))
)
display(df_silver.limit(5))

In [ ]:
# Write Silver Delta table, partitioned by genre
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("genre") \
    .saveAsTable("gaming_silver.vg_sales_clean")

spark.sql('OPTIMIZE gaming_silver.vg_sales_clean')
print("Silver table written and optimised ✅")
spark.sql('SELECT COUNT(*) AS silver_rows FROM gaming_silver.vg_sales_clean').show()

In [ ]:
# Sales tier distribution
display(spark.sql(
    'SELECT sales_tier, COUNT(*) AS titles, '
    'ROUND(SUM(global_sales_millions),2) AS total_sales_M '
    'FROM gaming_silver.vg_sales_clean '
    'GROUP BY sales_tier ORDER BY total_sales_M DESC'
))